# 01 — Data Exploration

Explores the raw Olist tables before any cleaning is applied. Goal: understand grain, keys, cardinality, and data-quality issues so that Phase 3 (cleaning) and metric definitions are built on a correct join model — not on assumptions.

No cleaning, KPI calculation, or dashboard work happens here.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from src.data_loader import load_raw_tables

pd.set_option("display.max_columns", None)

tables = load_raw_tables()
{name: df.shape for name, df in tables.items()}

{'customers': (99441, 5),
 'orders': (99441, 8),
 'order_items': (112650, 7),
 'order_payments': (103886, 5),
 'order_reviews': (99224, 7),
 'products': (32951, 9),
 'sellers': (3095, 4),
 'geolocation': (1000163, 5),
 'category_translation': (71, 2)}

## 1. Table overview

Shape, dtypes, duplicate rows, and missing values for every raw table.

In [2]:
for name, df in tables.items():
    print(f"--- {name}: {df.shape[0]:,} rows x {df.shape[1]} cols, {df.duplicated().sum():,} exact duplicate rows ---")
    missing = df.isna().sum()
    missing = missing[missing > 0]
    if len(missing):
        print((missing / len(df) * 100).round(2).astype(str) + "% missing")
    print()

--- customers: 99,441 rows x 5 cols, 0 exact duplicate rows ---

--- orders: 99,441 rows x 8 cols, 0 exact duplicate rows ---
order_approved_at                0.16% missing
order_delivered_carrier_date     1.79% missing
order_delivered_customer_date    2.98% missing
dtype: str

--- order_items: 112,650 rows x 7 cols, 0 exact duplicate rows ---

--- order_payments: 103,886 rows x 5 cols, 0 exact duplicate rows ---

--- order_reviews: 99,224 rows x 7 cols, 0 exact duplicate rows ---
review_comment_title      88.34% missing
review_comment_message     58.7% missing
dtype: str

--- products: 32,951 rows x 9 cols, 0 exact duplicate rows ---
product_category_name         1.85% missing
product_name_lenght           1.85% missing
product_description_lenght    1.85% missing
product_photos_qty            1.85% missing
product_weight_g              0.01% missing
product_length_cm             0.01% missing
product_height_cm             0.01% missing
product_width_cm              0.01% missing
dtype

**Observations**

- All tables are duplicate-free at the row level *except* `geolocation`, which has ~262k exact duplicate rows (26% of the table) — expected, since it's a many-rows-per-zip lookup table, not a deduplicated reference table.
- `order_reviews` has heavy missingness in free-text fields (`review_comment_title` 88%, `review_comment_message` 59%) — normal, since most reviews are a score with no comment.
- `products` has ~1.85% missing category and descriptive fields, consistent with the same ~610 rows.
- `orders` has missing values only in the delivery-lifecycle timestamp columns, which is expected for orders that haven't reached that stage yet (see §5).

## 2. Customers: `customer_id` vs `customer_unique_id`

This is the most important modelling decision in the dataset. Olist assigns a **new `customer_id` to every order**, even for the same person. `customer_unique_id` is the actual, stable identifier for a person across orders.

In [3]:
c = tables["customers"]
print("customers rows:", len(c))
print("distinct customer_id:", c["customer_id"].nunique())
print("distinct customer_unique_id:", c["customer_unique_id"].nunique())

repeat_people = c["customer_unique_id"].value_counts()
print("customer_unique_id values linked to >1 customer_id:", (repeat_people > 1).sum())

customers rows: 99441
distinct customer_id: 99441
distinct customer_unique_id: 96096
customer_unique_id values linked to >1 customer_id: 2997


**Conclusion:** `customer_id` is 1:1 with a row in `customers` (and, in effect, 1:1 with an order) — it is **not** a valid key for repeat-purchase analysis. `customer_unique_id` is the correct key for anything involving "the same customer buying more than once" (repeat rate, RFM, cohorts). This will be a core rule in Phase 3.

## 3. Orders: status and timestamp lifecycle

In [4]:
o = tables["orders"]
print("orders rows:", len(o), "| distinct order_id:", o["order_id"].nunique())
print()
print(o["order_status"].value_counts())

orders rows: 99441 | distinct order_id: 99441

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [5]:
ts_cols = [c for c in o.columns if "date" in c or "timestamp" in c]
o_ts = o.copy()
for col in ts_cols:
    o_ts[col] = pd.to_datetime(o_ts[col], errors="coerce")

print("Missing values per timestamp column:")
print(o_ts[ts_cols].isna().sum())
print()
print("Date range (order_purchase_timestamp):", o_ts["order_purchase_timestamp"].min(), "to", o_ts["order_purchase_timestamp"].max())

Missing values per timestamp column:
order_purchase_timestamp            0
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Date range (order_purchase_timestamp): 2016-09-04 21:15:19 to 2018-10-17 17:30:18


In [6]:
# Sanity check: can an order be "delivered" before it was purchased?
impossible = o_ts[o_ts["order_delivered_customer_date"] < o_ts["order_purchase_timestamp"]]
print("Orders delivered before purchase (should be 0):", len(impossible))

# Orders with status = delivered but no delivered_customer_date (data-quality anomaly)
missing_delivered = o_ts[(o_ts["order_status"] == "delivered") & (o_ts["order_delivered_customer_date"].isna())]
print("Orders marked 'delivered' with a missing delivery date:", len(missing_delivered))

Orders delivered before purchase (should be 0): 0
Orders marked 'delivered' with a missing delivery date: 8


**Observations**

- `order_status` has 8 values. 96,478 orders (97%) are `delivered`; the rest span the pipeline (`shipped`, `canceled`, `unavailable`, `invoiced`, `processing`, `created`, `approved`).
- Missing delivery timestamps track order status almost perfectly — an order that's `shipped`, `canceled`, `unavailable`, `invoiced`, or `processing` simply hasn't reached that lifecycle stage yet, so the missingness is *structural*, not a data-quality defect.
- 8 orders are marked `delivered` but have no `order_delivered_customer_date` — a small (0.008% of orders) genuine data-quality anomaly to exclude or flag in Phase 3.
- No orders show an impossible delivered-before-purchased timestamp.
- `order_estimated_delivery_date` is never missing, so it's safe to use for every order regardless of status.

## 4. Order items: grain and cardinality

**This is the critical double-counting risk in the dataset.** `order_items` has one row per item within an order — an order with 3 items contributes 3 rows.

In [7]:
oi = tables["order_items"]
print("order_items rows:", len(oi))
print("distinct order_id:", oi["order_id"].nunique())

items_per_order = oi.groupby("order_id").size()
print()
print(items_per_order.describe())
print()
pct_multi = (items_per_order > 1).mean() * 100
print(f"Orders with more than one item: {(items_per_order > 1).sum():,} ({pct_multi:.1f}%)")

order_items rows: 112650
distinct order_id: 98666

count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
dtype: float64

Orders with more than one item: 9,803 (9.9%)


**Rule for Phase 3:** revenue and item-level metrics must aggregate `order_items` to the `order_id` grain (e.g. `sum(price)` per order) *before* joining to anything else that isn't also at item grain — otherwise joins will fan out and inflate totals (demonstrated in §8).

## 5. Payments: multiple rows per order

Payments are **not** 1:1 with orders — a single order can be paid with a combination of methods (e.g. voucher + credit card), each producing its own payment row.

In [8]:
pay = tables["order_payments"]
print("order_payments rows:", len(pay))
print("distinct order_id:", pay["order_id"].nunique(), "vs orders table:", o["order_id"].nunique())

pay_per_order = pay.groupby("order_id").size()
print()
print(f"Orders with more than one payment row: {(pay_per_order > 1).sum():,}")
print("Max payment rows for a single order:", pay_per_order.max())
print()
print(pay["payment_type"].value_counts())

order_payments rows: 103886
distinct order_id: 99440 vs orders table: 99441

Orders with more than one payment row: 2,961
Max payment rows for a single order: 29

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


In [9]:
print("payment_value <= 0:", (pay["payment_value"] <= 0).sum(), "rows — suspicious, investigate in Phase 3.")

payment_value <= 0: 9 rows — suspicious, investigate in Phase 3.


**Rule for Phase 3:** `payment_value` must be summed per `order_id` before use — a raw join of `order_items` to `order_payments` on `order_id` fans out on **both** sides and is a direct double-counting trap (see §8). Also note 1 order in `orders` has zero payment rows.

## 6. Reviews: grain and score distribution

In [10]:
rev = tables["order_reviews"]
print("order_reviews rows:", len(rev))
print("distinct order_id:", rev["order_id"].nunique())
print("distinct review_id:", rev["review_id"].nunique())

rev_per_order = rev.groupby("order_id").size()
print("Orders with more than one review row:", (rev_per_order > 1).sum())

orders_without_review = o["order_id"].nunique() - rev["order_id"].nunique()
print("Orders with no review at all:", orders_without_review)
print()
print(rev["review_score"].value_counts().sort_index())

order_reviews rows: 99224
distinct order_id: 98673
distinct review_id: 98410
Orders with more than one review row: 547
Orders with no review at all: 768

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64


**Observations**

- A small number of orders (547) have more than one review row — likely a re-review after a follow-up. `review_id` is not a safe primary key on its own (98,410 distinct vs 99,224 rows).
- 768 orders have no review at all. These should be treated as **missing**, not assumed to be a poor score — averaging review scores must use only orders that have a review, with the missing rate reported alongside.
- Scores skew strongly positive (57,328 are 5-star; 11,424 are 1-star), typical for e-commerce review data.

## 7. Products and category translation

In [11]:
prod = tables["products"]
print("products rows:", len(prod), "| distinct product_id:", prod["product_id"].nunique())
print("missing product_category_name:", prod["product_category_name"].isna().sum(),
      f"({prod['product_category_name'].isna().mean()*100:.2f}%)")

trans = tables["category_translation"]
cats_in_products = set(prod["product_category_name"].dropna().unique())
cats_in_translation = set(trans["product_category_name"].unique())
untranslated = cats_in_products - cats_in_translation
print("Distinct categories in products:", len(cats_in_products))
print("Categories with no English translation available:", untranslated)

products rows: 32951 | distinct product_id: 32951
missing product_category_name: 610 (1.85%)
Distinct categories in products: 73
Categories with no English translation available: {'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}


**Rule for Phase 3:** 610 products (1.85%) have no category at all — keep them as an explicit `"unknown"` category rather than dropping the rows, since they still carry real revenue. Two categories (`pc_gamer`, `portateis_cozinha_e_preparadores_de_alimentos`) exist in `products` but have no row in the translation table — fall back to the original Portuguese name for those rather than dropping or erroring.

## 8. Sellers and geolocation

In [12]:
sellers = tables["sellers"]
print("sellers rows:", len(sellers), "| distinct seller_id:", sellers["seller_id"].nunique())

geo = tables["geolocation"]
print()
print("geolocation rows:", len(geo))
print("distinct zip_code_prefix:", geo["geolocation_zip_code_prefix"].nunique())
rows_per_zip = geo.groupby("geolocation_zip_code_prefix").size()
print("rows per zip prefix — mean:", round(rows_per_zip.mean(), 1), "| max:", rows_per_zip.max())

sellers rows: 3095 | distinct seller_id: 3095

geolocation rows: 1000163
distinct zip_code_prefix: 19015
rows per zip prefix — mean: 52.6 | max: 1146


**Rule for Phase 3:** `geolocation` is a noisy many-rows-per-zip-prefix lookup (avg ~53 lat/lng pairs per prefix, some with duplicate rows), **not** a clean dimension table. If used for mapping, it must be aggregated to one row per `zip_code_prefix` (e.g. median lat/lng) before joining — joining it directly to `customers` or `sellers` on zip prefix would fan out rows massively.

## 9. Join integrity checks

Confirms referential integrity before relying on these joins for Phase 3 modelling.

In [13]:
checks = {
    "order_items -> products (missing product)": oi.merge(prod[["product_id"]], on="product_id", how="left", indicator=True).query("_merge == 'left_only'").shape[0],
    "order_items -> sellers (missing seller)": oi.merge(sellers[["seller_id"]], on="seller_id", how="left", indicator=True).query("_merge == 'left_only'").shape[0],
    "orders -> customers (missing customer)": o.merge(c[["customer_id"]], on="customer_id", how="left", indicator=True).query("_merge == 'left_only'").shape[0],
}
checks

{'order_items -> products (missing product)': 0,
 'order_items -> sellers (missing seller)': 0,
 'orders -> customers (missing customer)': 0}

Every foreign key checked resolves cleanly — no orphaned `product_id`, `seller_id`, or `customer_id` references. Referential integrity is not a concern for this dataset; grain/cardinality is.

## 10. Double-counting demonstration

The clearest illustration of the join-cardinality risk flagged in §4 and §5: naively joining `order_items` directly to `order_payments` on `order_id` fans out on both sides and inflates the revenue total.

In [14]:
correct_revenue = oi["price"].sum()
print(f"Correct item revenue — sum(order_items.price), no join: R$ {correct_revenue:,.2f}")

naive = oi.merge(pay, on="order_id")
print(f"Row count after naive order_items x order_payments join: {len(naive):,} (order_items alone: {len(oi):,})")
print(f"Naive sum(price) after that join (inflated, WRONG): R$ {naive['price'].sum():,.2f}")

Correct item revenue — sum(order_items.price), no join: R$ 13,591,643.70
Row count after naive order_items x order_payments join: 117,601 (order_items alone: 112,650)
Naive sum(price) after that join (inflated, WRONG): R$ 14,209,115.34


**This is exactly the mistake Phase 3 must avoid.** The fix: aggregate `order_items` to one row per `order_id` (item count, item revenue) and `order_payments` to one row per `order_id` (total paid) *independently*, then join those two order-grain tables together — never join item-grain and payment-grain tables directly.

## Summary: relationships and Phase 3 decisions

**Grain of each table**

| Table | Grain | Notes |
|---|---|---|
| `customers` | 1 row per order-customer link | Use `customer_unique_id` for "the same person", not `customer_id` |
| `orders` | 1 row per order | Core fact table for order-level metrics |
| `order_items` | 1 row per item within an order | Must aggregate to order grain before joining to payments |
| `order_payments` | 1+ rows per order | Multiple rows = split payment methods; sum per order before joining |
| `order_reviews` | ~1 row per order (some orders have 2+, some have 0) | Missing ≠ bad score; treat as missing data |
| `products` | 1 row per product | 1.85% missing category — keep as "unknown", don't drop |
| `sellers` | 1 row per seller | Clean |
| `geolocation` | many rows per zip prefix | Aggregate to 1 row per prefix before any geo join |
| `product_category_name_translation` | 1 row per category | 2 categories in `products` have no translation |

**Confirmed relationships**

- `orders.customer_id` → `customers.customer_id` (1:1, fully resolved)
- `customers.customer_unique_id` identifies a real person across multiple `customer_id`/orders (2,997 people have placed more than one order)
- `order_items.order_id` → `orders.order_id` (many:1)
- `order_items.product_id` → `products.product_id` (many:1, fully resolved)
- `order_items.seller_id` → `sellers.seller_id` (many:1, fully resolved)
- `order_payments.order_id` → `orders.order_id` (many:1)
- `order_reviews.order_id` → `orders.order_id` (mostly 1:1, a few duplicates)
- `products.product_category_name` → `product_category_name_translation.product_category_name` (many:1, 2 unmatched categories)

**Key risk:** joining `order_items` directly to `order_payments` (or `order_reviews`) on `order_id` fans out both sides and silently inflates revenue and order counts. Every metric touching more than one item-or-payment-grain table must aggregate to order grain first.

**Decisions carried into Phase 3**

1. Deduplicate customers to `customer_unique_id` for any customer-level or repeat-purchase metric; keep `customer_id` only for joining to `orders`.
2. Build an order-grain `orders_clean` table with one row per `order_id`: item count, item revenue (`sum(price)`), freight (`sum(freight_value)`), total paid (`sum(payment_value)` from a separately-aggregated payments table), and delivery-delay fields.
3. Define which `order_status` values count as "valid" for revenue/order KPIs before Phase 4 (likely `delivered`, documented explicitly either way — `canceled`/`unavailable` orders should not count as revenue).
4. Keep the 610 products with no category as `"unknown"` rather than dropping them; fall back to the Portuguese name for the 2 categories missing an English translation.
5. Aggregate `geolocation` to one row per `zip_code_prefix` (e.g. median lat/lng) before any geographic join.
6. Treat missing reviews as missing, not as a 0 or negative signal, when computing average review score.
7. Flag (not silently drop) the 8 orders marked `delivered` with no delivery timestamp, and the 9 payment rows with `payment_value <= 0`, during cleaning.